# Anatomy of an Effective Prompt

Source slide: `slides/prompt-engineering/03_Anatomy_of_an_Effective_Prompt.md`

Each technique here matches one named component from the slide.



## Prerequisites

- From the repository root, install all notebook and test prerequisites: `pip install -r requirements-dev.txt`
- Sign in to GitHub Copilot in VS Code or GitHub Codespaces
- These examples use ambient auth; do not paste a PAT into the notebook

Each technique below has a failure cell and an improved cell that you can run independently with the real GitHub Copilot SDK.


In [ ]:
from copilot import CopilotClient
from copilot.session import PermissionHandler

model = "gpt-4.1"

async def ask_copilot(
    prompt: str,
    *,
    system_message: str | None = None,
    model_name: str = model,
) -> str:
    """Run a single prompt through the real GitHub Copilot SDK using ambient auth."""
    async with CopilotClient() as client:
        session_kwargs = {
            "model": model_name,
            "on_permission_request": PermissionHandler.approve_all,
        }
        if system_message:
            session_kwargs["system_message"] = {
                "mode": "append",
                "content": system_message,
            }
        async with await client.create_session(**session_kwargs) as session:
            response = await session.send_and_wait(prompt)
            return response.data.content if response and response.data else ""

def show_block(title: str, content: str) -> None:
    print(title)
    print("=" * 80)
    print(content.strip())
    print()

print("✅ Setup complete - real GitHub Copilot SDK with ambient auth")
print(f"📦 Using model: {model}")


## Core Task

**Failure mode:** The core task is the non-negotiable instruction. When it is vague, the model has to invent the goal before it can solve it.

**Failure test:** The failure prompt asks for “help with onboarding,” which leaves audience, deliverable, and scope unstated. That usually produces generic advice instead of the specific artifact you wanted.

**Failure example:** Run the next cell by itself to execute the weaker prompt in isolation and compare the returned result to the failure test above.

**Technique:** State the exact job, audience, and deliverable in direct language.

**Improved example:** The improved prompt anchors the task to a short onboarding summary for new admins and narrows the topic to SSO setup, so the model can target the right output. Run the later cell by itself to execute the improved prompt in isolation and inspect the stronger result.


In [ ]:
failure_prompt = 'Help with onboarding.'
show_block("❌ Failure prompt", failure_prompt)
failure_result = await ask_copilot(
    failure_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("❌ Failure result", failure_result)
failure_test = 'The failure prompt asks for “help with onboarding,” which leaves audience, deliverable, and scope unstated. That usually produces generic advice instead of the specific artifact you wanted.'
show_block("🧪 Why this fails", failure_test)


In [ ]:
improved_prompt = 'Write a 3-bullet onboarding summary for new workspace admins. Focus only on the first-day SSO setup steps and keep each bullet to one sentence.'
show_block("✅ Improved prompt", improved_prompt)
improved_result = await ask_copilot(
    improved_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("✅ Improved result", improved_result)
fix_explanation = 'The improved prompt anchors the task to a short onboarding summary for new admins and narrows the topic to SSO setup, so the model can target the right output.'
show_block("🔧 Why this is better", fix_explanation)


## System Instructions

**Failure mode:** System instructions shape the model’s behavior before the task begins. Beginners often skip them and then wonder why tone, format, or rigor changes from answer to answer.

**Failure test:** The failure prompt requests a release-note style explanation but provides no role or formatting guardrails, so the model may default to long-form explanatory prose.

**Failure example:** Run the next cell by itself to execute the weaker prompt in isolation and compare the returned result to the failure test above.

**Technique:** Add a system instruction that sets role, tone, and formatting constraints before the user prompt runs.

**Improved example:** The improved run uses a system instruction that frames the model as a release-note editor, which makes concise, reviewer-friendly output much more likely. Run the later cell by itself to execute the improved prompt in isolation and inspect the stronger result.


In [ ]:
failure_prompt = 'Explain the latest API authentication change for customers.'
show_block("❌ Failure prompt", failure_prompt)
failure_result = await ask_copilot(
    failure_prompt,
    system_message='You are a release note editor. Be concise, factual, and customer-facing. Output exactly two bullets.',
)
show_block("❌ Failure result", failure_result)
failure_test = 'The failure prompt requests a release-note style explanation but provides no role or formatting guardrails, so the model may default to long-form explanatory prose.'
show_block("🧪 Why this fails", failure_test)


In [ ]:
improved_prompt = 'Explain the latest API authentication change for customers in two short bullets.'
show_block("✅ Improved prompt", improved_prompt)
improved_result = await ask_copilot(
    improved_prompt,
    system_message='You are a release note editor. Be concise, factual, and customer-facing. Output exactly two bullets.',
)
show_block("✅ Improved result", improved_result)
fix_explanation = 'The improved run uses a system instruction that frames the model as a release-note editor, which makes concise, reviewer-friendly output much more likely.'
show_block("🔧 Why this is better", fix_explanation)


## Examples

**Failure mode:** Examples demonstrate the target pattern. Without them, the model may complete the task correctly in spirit while still missing the exact format you need.

**Failure test:** The failure prompt asks for a changelog entry but never shows the desired structure, so the model can invent its own style.

**Failure example:** Run the next cell by itself to execute the weaker prompt in isolation and compare the returned result to the failure test above.

**Technique:** Include at least one input-output example that demonstrates the required pattern.

**Improved example:** The improved prompt includes one sample changelog entry, which gives the model a concrete format to imitate on the new item. Run the later cell by itself to execute the improved prompt in isolation and inspect the stronger result.


In [ ]:
failure_prompt = 'Turn this issue into a changelog entry: “Added bulk user import with CSV validation errors.”'
show_block("❌ Failure prompt", failure_prompt)
failure_result = await ask_copilot(
    failure_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("❌ Failure result", failure_result)
failure_test = 'The failure prompt asks for a changelog entry but never shows the desired structure, so the model can invent its own style.'
show_block("🧪 Why this fails", failure_test)


In [ ]:
improved_prompt = 'Convert product updates into changelog entries.\n\nIssue: “Added webhook retry diagnostics.”\nChangelog: “Added webhook retry diagnostics so operators can inspect failed delivery attempts faster.”\n\nIssue: “Added bulk user import with CSV validation errors.”\nChangelog:'
show_block("✅ Improved prompt", improved_prompt)
improved_result = await ask_copilot(
    improved_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("✅ Improved result", improved_result)
fix_explanation = 'The improved prompt includes one sample changelog entry, which gives the model a concrete format to imitate on the new item.'
show_block("🔧 Why this is better", fix_explanation)


## Contextual Information

**Failure mode:** Contextual information gives the model the facts it should ground itself in. If you omit the source material, the model will rely on prior knowledge or guesses.

**Failure test:** The failure prompt asks for a summary of a customer interview without including the notes, so the model has no trustworthy evidence to summarize.

**Failure example:** Run the next cell by itself to execute the weaker prompt in isolation and compare the returned result to the failure test above.

**Technique:** Attach the relevant notes, documents, or tables directly in the prompt.

**Improved example:** The improved prompt embeds the interview notes, so the answer can stay tied to the actual evidence instead of inventing generic findings. Run the later cell by itself to execute the improved prompt in isolation and inspect the stronger result.


In [ ]:
failure_prompt = 'Summarize the customer interview and identify the main friction points.'
show_block("❌ Failure prompt", failure_prompt)
failure_result = await ask_copilot(
    failure_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("❌ Failure result", failure_result)
failure_test = 'The failure prompt asks for a summary of a customer interview without including the notes, so the model has no trustworthy evidence to summarize.'
show_block("🧪 Why this fails", failure_test)


In [ ]:
improved_prompt = 'Based only on these interview notes, summarize the customer interview and list the top two friction points.\n\nInterview notes:\n- Admin setup took 45 minutes because SSO metadata was hard to locate.\n- The user liked the dashboard once setup was complete.\n- CSV import errors were unclear and had to be retried twice.\n- Billing export worked without issues.'
show_block("✅ Improved prompt", improved_prompt)
improved_result = await ask_copilot(
    improved_prompt,
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
show_block("✅ Improved result", improved_result)
fix_explanation = 'The improved prompt embeds the interview notes, so the answer can stay tied to the actual evidence instead of inventing generic findings.'
show_block("🔧 Why this is better", fix_explanation)
